# Loopo Italian FunctionGemma experiment

Use **Runtime → Change runtime type → T4 GPU** for the two-step smoke and full runs. On Hugging Face, accept the [FunctionGemma license](https://huggingface.co/google/functiongemma-270m-it), then grant this notebook access to a read token named **HF_TOKEN** in Colab Secrets. Run every cell in order. The default experiment scores a small validation slice and trains for two optimizer steps.

## 1. Environment and GPU diagnostics

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Select a Colab runtime with Python 3.11–3.13.")

print(sys.version)
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    check=False,
).returncode == 0
print("GPU available:", has_gpu)

## 2. Choose experiment, checkout and persistence

Use the ref containing this notebook. Keep `main` after merge or enter the feature branch or a commit SHA before merge. Use a unique run name for each fresh run. Set `resume = True` only for a saved checkpoint with identical settings. Mount Drive to keep checkpoints when Colab disconnects.

For the numeric follow-up, choose **contrastive**, enable **use_drive**, and keep the full commit SHA and run name for recovery. The profile always exports a merged model and a verified archive.

In [ ]:
#@title Experiment options
repo_ref = "main" #@param {type:"string"}
experiment = "smoke" #@param ["smoke", "seed", "balanced", "expanded", "base", "contrastive", "targeted"]
use_drive = False #@param {type:"boolean"}
run_name = "trial-01" #@param {type:"string"}
resume = False #@param {type:"boolean"}
export_merged = False #@param {type:"boolean"}
examples_file = "" #@param {type:"string"}
initial_adapter_path = "/content/drive/MyDrive/loopo-functiongemma/contrastive/contrastive-2026-11-18-01/training/checkpoint-1125" #@param {type:"string"}

persistent_experiment = experiment in {"contrastive", "targeted"}
if persistent_experiment and not use_drive:
    raise ValueError("This experiment requires Drive for recovery.")
if not has_gpu and experiment != "smoke":
    raise RuntimeError("Select a GPU runtime for full training or held-out replay.")
if not run_name or Path(run_name).name != run_name or run_name in {".", ".."}:
    raise ValueError("Use a single directory name for run_name.")

checkout = Path("/content/loopo")
if not checkout.exists():
    subprocess.run(["git", "clone", "https://github.com/p33nah/loopo.git", str(checkout)], check=True)
subprocess.run(["git", "-C", str(checkout), "fetch", "origin", repo_ref], check=True)
subprocess.run(["git", "-C", str(checkout), "switch", "--detach", "FETCH_HEAD"], check=True)
project = checkout / "ml/functiongemma"
if not (project / "pyproject.toml").is_file():
    raise RuntimeError("Choose a repo_ref containing ml/functiongemma.")

results_root = Path("/content/loopo-runs")
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir() or not os.path.ismount(my_drive.parent):
        raise RuntimeError("Drive is not mounted. Reconnect it before continuing.")
    results_root = my_drive / "loopo-functiongemma"
run = results_root / experiment / run_name
if not resume and run.exists() and any(run.iterdir()):
    raise ValueError("Run exists. Enable resume or choose a fresh name.")
run.mkdir(parents=True, exist_ok=True)
initial_adapter_source = Path(initial_adapter_path) if experiment == "targeted" else None
initial_adapter = run / "parent-checkpoint/adapter" if initial_adapter_source is not None else None
print("Artifacts:", run)

## 3. Install pinned training dependencies

The notebook uses a virtual environment so the training packages do not replace Colab's kernel packages.

In [ ]:
venv = Path("/content/loopo-functiongemma-venv")
subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
python = str(venv / "bin/python")
subprocess.run([sys.executable, "-m", "pip", "--python", python, "install", "pip"], check=True)
wheel_index = "cu128" if has_gpu else "cpu"
subprocess.run(
    [python, "-m", "pip", "install", "torch==2.10.0",
     "--index-url", f"https://download.pytorch.org/whl/{wheel_index}"],
    check=True,
)
subprocess.run([python, "-m", "pip", "install", "-e", str(project) + "[ml]"], check=True)
subprocess.run([python, "-m", "pip", "check"], check=True)
subprocess.run(
    [python, "-c", "import torch; print(torch.__version__, torch.version.cuda); "
     f"assert torch.cuda.is_available() == {has_gpu!r}"],
    check=True,
)

## 4. Authenticate with Hugging Face

The read token is passed to the ML subprocesses through their environment.

In [ ]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.")
ml_env = {**os.environ, "HF_TOKEN": hf_token}
del hf_token

def cli(module, *args):
    subprocess.run(
        [python, "-m", "loopo_functiongemma." + module, *map(str, args)],
        cwd=project, env=ml_env, check=True,
    )

## 5. Generate and validate the dataset and tool schemas

In [ ]:
import hashlib
import tempfile

data = run / "data"
config_name = {"smoke": "smoke.toml", "contrastive": "contrastive.toml", "targeted": "redmi-targeted.toml"}.get(experiment, "train.toml")
config = run / "train.toml"

def publish_input(source, target):
    if target.exists():
        if target.read_bytes() != source.read_bytes():
            raise ValueError(f"Run input changed: {target.name}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    pending = target.with_suffix(target.suffix + ".part")
    shutil.copyfile(source, pending)
    pending.replace(target)

initial_inputs = []
if initial_adapter is not None:
    initial_files = (
        (initial_adapter_source.parent / "run.json", initial_adapter.parent / "run.json"),
        (initial_adapter_source / "adapter_config.json", initial_adapter / "adapter_config.json"),
        (initial_adapter_source / "adapter_model.safetensors", initial_adapter / "adapter_model.safetensors"),
    )
    for source, target in initial_files:
        if source.is_file():
            publish_input(source, target)
        elif not (resume and target.is_file()):
            raise FileNotFoundError(f"Targeted training needs the parent checkpoint file: {source}")
        initial_inputs.append(target)

with tempfile.TemporaryDirectory() as temporary:
    generated = Path(temporary) / "data"
    if experiment == "targeted":
        cli("corpus", "import", generated,
            "--examples", project / "examples/redmi-targeted.txt",
            "--validation", project / "examples/redmi-targeted-validation.txt",
            "--holdout", project / "loopo_functiongemma/examples/numeric-replay.txt",
            "--holdout", project / "reports/rejection-probe.txt",
            "--holdout", project / "reports/contrastive-holdout.txt",
            "--holdout", project / "reports/targeted-holdout.txt")
        shutil.copyfile(project / "reports/targeted-holdout.jsonl", generated / "targeted-holdout.jsonl")
        subprocess.run([python, "-c",
                        "import sys; from pathlib import Path; "
                        "from loopo_functiongemma.corpus import EXAMPLES, read_examples; "
                        "from loopo_functiongemma.dataset import _write_jsonl; "
                        "_write_jsonl(Path(sys.argv[1]), read_examples(EXAMPLES / 'numeric-replay.txt', 'heldout_numeric-replay'))",
                        str(generated / "numeric-replay.jsonl")], cwd=project, check=True)
    elif experiment in {"balanced", "contrastive"}:
        extra = ["--examples", examples_file] if examples_file else []
        if experiment == "contrastive":
            extra += ["--profile", "contrastive", "--holdout", project / "reports/rejection-probe.txt",
                      "--holdout", project / "reports/contrastive-holdout.txt",
                      "--examples", project / "examples/rejection-followup.txt"]
        cli("corpus", "generate", generated, *extra)
    else:
        expansion = ["--expand-train", "2500"] if experiment == "expanded" else []
        cli("dataset", "generate", generated, *expansion)
    for source in sorted(generated.iterdir()):
        publish_input(source, data / source.name)

    publish_input(project / "configs" / config_name, config)
    publish_input(project / "notebooks/loopo_functiongemma_finetuning.ipynb", run / "notebook.ipynb")
    provenance = Path(temporary) / "inputs.json"
    revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    provenance.write_text(json.dumps({
        "source_commit": revision,
        "profile": experiment,
        "inputs_sha256": {
            str(path.relative_to(run)): hashlib.sha256(path.read_bytes()).hexdigest()
            for path in [config, *initial_inputs, *sorted(data.glob("*.jsonl"))]
        },
    }, indent=2) + "\n", encoding="utf-8")
    publish_input(provenance, run / provenance.name)

cli("dataset", "validate", data)
environment = run / "packages.json"
if not environment.exists():
    with tempfile.TemporaryDirectory() as temporary:
        snapshot = Path(temporary) / environment.name
        with snapshot.open("w", encoding="utf-8") as file:
            subprocess.run([python, "-m", "pip", "list", "--format=json"], stdout=file, check=True)
        publish_input(snapshot, environment)

## 6. Load the official tokenizer and verify native formatting

This step checks that every target begins after the inference prompt, fits the sequence length and round-trips to the expected ordered plan.

In [ ]:
cli("model", "validate-format", data)

## 7. Evaluate the base model

The smoke run uses four validation rows. Full and base-only experiments score the frozen test and challenge sets. The targeted experiment measures the candidate after validation has selected its checkpoint. Raw predictions are saved for the shared benchmark.

In [ ]:
if experiment == "smoke":
    sample = run / "smoke-validation.jsonl"
    lines = (data / "validation.jsonl").read_text(encoding="utf-8").splitlines()
    sample.write_text("\n".join(lines[:4]) + "\n", encoding="utf-8")
    benchmark_sets = {"smoke-validation": sample}
else:
    benchmark_sets = {split: data / f"{split}.jsonl" for split in ("test", "challenge")}

if experiment == "contrastive":
    benchmark_sets.update({name: data / f"{name}.jsonl" for name in ("numeric-replay", "rejection-probe", "contrastive-holdout")})

if experiment == "targeted":
    benchmark_sets.update({name: data / f"{name}.jsonl" for name in ("numeric-replay", "targeted-holdout")})

def predict(input_file, output, *model_args):
    if output.is_file():
        try:
            saved = json.loads(output.read_text(encoding="utf-8"))["metadata"]
            identity = {"adapter": None, "merged": None}
            if model_args:
                mode, path = model_args
                identity[mode.removeprefix("--")] = str(path)
            if saved["dataset_sha256"] != hashlib.sha256(input_file.read_bytes()).hexdigest() or any(
                saved.get(key) != value for key, value in identity.items()
            ):
                raise ValueError(f"Prediction data or model changed: {output.name}")
            return
        except json.JSONDecodeError:
            pass
    pending = output.with_suffix(".part")
    cli("evaluate", "predict", input_file, pending, *model_args)
    pending.replace(output)

if experiment != "targeted":
    for name, input_file in benchmark_sets.items():
        predict(input_file, run / f"{name}-base.json")

## 8. Fine-tune LoRA, select a checkpoint and save the adapter

The trainer uses validation loss to select the checkpoint. Compare training and validation loss in `training-metrics.json` or TensorBoard. For a new run choose a fresh run name. For a resume retain the same ref, dataset, config, GPU type and environment.

In [ ]:
training = run / "training"
metrics_path = training / "training-metrics.json"
if experiment != "base":
    if not metrics_path.is_file():
        existing_training = training.exists() and any(training.iterdir())
        train_args = ["--resume"] if resume and existing_training else []
        if initial_adapter is not None:
            train_args += ["--initial-adapter", initial_adapter]
        cli("train", data, config, training, *train_args)
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("Validation-loss checkpoint:", metrics["best_checkpoint"])
    selected = training / "adapter"
    if persistent_experiment:
        epoch_steps = sorted({entry["step"] for entry in metrics["history"] if "eval_loss" in entry})
        if not epoch_steps:
            raise RuntimeError("Missing validation history")
        candidates = []
        for step in epoch_steps:
            checkpoint = training / f"checkpoint-{step}"
            subprocess.run([python, "-c", "from pathlib import Path; import sys; "
                            "from loopo_functiongemma.train import verify_checkpoint; "
                            "verify_checkpoint(Path(sys.argv[1]))", str(checkpoint)], cwd=project, check=True)
            output = run / f"validation-{checkpoint.name}.json"
            predict(data / "validation.jsonl", output, "--adapter", checkpoint)
            candidates.extend(("--input", f"{checkpoint.name}={output}"))
        selection = run / "checkpoint-selection.json"
        cli("evaluate", "select", data / "validation.jsonl", selection, *candidates)
        choice = json.loads(selection.read_text(encoding="utf-8"))
        selected = training / choice["selected"]
        print("Selected from validation predictions:", choice)

## 9. Export standalone weights and compare outputs

The contrastive profile exports the selected checkpoint even when the optional export is off. The merged model is evaluated separately so the prediction files precede any quantization.

In [ ]:
merged = run / "merged-hf"
if (export_merged or persistent_experiment) and experiment != "base":
    if not merged.exists():
        with tempfile.TemporaryDirectory(prefix=f".{run.name}-merge-", dir=run.parent) as temporary:
            candidate = Path(temporary) / "model"
            cli("model", "merge", selected, candidate)
            candidate.rename(merged)
    export = json.loads((merged / "loopo-export.json").read_text(encoding="utf-8"))
    if Path(export["adapter"]) != selected:
        raise ValueError("Existing merged model was exported from a different checkpoint.")

for name, input_file in benchmark_sets.items():
    report_inputs = []
    if experiment != "targeted":
        report_inputs += ["--input", f"base={run / (name + '-base.json')}"]
    if experiment != "base":
        adapter_results = run / f"{name}-tuned.json"
        predict(input_file, adapter_results, "--adapter", selected)
        report_inputs += ["--input", f"tuned={adapter_results}"]
    if experiment == "contrastive":
        merged_results = run / f"{name}-merged.json"
        predict(input_file, merged_results, "--merged", merged)
        report_inputs += ["--input", f"merged={merged_results}"]
    cli("evaluate", "compare", input_file, run / f"{name}-report", *report_inputs)
    print((run / f"{name}-report" / "summary.md").read_text(encoding="utf-8"))

## 10. Verify a portable archive

Keep the Drive directory for interruption recovery and a verified ZIP for a second copy. For INT8 conversion, follow the Android guide using the retained merged model. Archive again with a new ZIP name once the bundle and its results are in the run directory.

In [ ]:
if persistent_experiment:
    required = [
        run / "inputs.json", run / "packages.json", run / "checkpoint-selection.json",
        training / "adapter/adapter_model.safetensors", training / "training-metrics.json",
        merged / "loopo-export.json", merged / "tokenizer.model",
        *[run / f"{name}-report/report.json" for name in benchmark_sets],
    ]
    if not all(path.is_file() and path.stat().st_size for path in required) or not any(merged.glob("*.safetensors")):
        raise RuntimeError("Complete the missing stages before archiving this run.")
    subprocess.run([python, "-c", "from pathlib import Path; import sys; "
                    "from loopo_functiongemma.train import verify_checkpoint; "
                    "[verify_checkpoint(path) for path in Path(sys.argv[1]).glob('checkpoint-*')]",
                    str(training)], cwd=project, check=True)
if use_drive:
    archive = run.parent / f"{run.name}.zip"
    cli("artifacts", "archive", run, archive)
    cli("artifacts", "verify", archive)
    print("Verified archive:", archive)
print("Run path:", run)

## 11. Resume or copy the run

After a disconnect, open the same code ref and set the same run name, GPU and profile. Enable **resume**, then run the cells in order. A completed training stage is reused. Interrupted training resumes from its latest verified checkpoint. If training stopped before a complete checkpoint, use a new run name and keep the old directory for diagnosis.

Download the verified ZIP and its SHA-256 sidecar or copy both to another durable location. Keep the original directory to resume training.